In [1]:
#feature engineering
import pandas as pd
import numpy as np
DATA_PATH = '../data'

ratings = pd.read_csv(f'{DATA_PATH}/ratings.csv')
movies = pd.read_csv(f'{DATA_PATH}/movies.csv')

# Convert timestamp
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['hour'] = ratings['datetime'].dt.hour
ratings['day_of_week'] = ratings['datetime'].dt.dayofweek
ratings['month'] = ratings['datetime'].dt.month
ratings['year'] = ratings['datetime'].dt.year

print("Loaded successfully")
print(ratings.shape)

Loaded successfully
(25000095, 9)


In [2]:
# User features
user_features = ratings.groupby('userId').agg(
    user_rating_count=('rating', 'count'),
    user_avg_rating=('rating', 'mean'),
    user_rating_std=('rating', 'std'),
    user_peak_hour=('hour', lambda x: x.value_counts().idxmax()),
    user_peak_day=('day_of_week', lambda x: x.value_counts().idxmax()),
    user_peak_season=('month', lambda x: (
        'winter' if x.value_counts().idxmax() in [12, 1, 2]
        else 'spring' if x.value_counts().idxmax() in [3, 4, 5]
        else 'summer' if x.value_counts().idxmax() in [6, 7, 8]
        else 'fall'
    ))
).reset_index()

print(user_features.head())
print(user_features.shape)

   userId  user_rating_count  user_avg_rating  user_rating_std  \
0       1                 70         3.814286         1.004235   
1       2                184         3.630435         1.457728   
2       3                656         3.697409         0.599854   
3       4                242         3.378099         1.116927   
4       5                101         3.752475         0.931729   

   user_peak_hour  user_peak_day user_peak_season  
0              12              2           spring  
1              20              4           spring  
2              13              3           summer  
3              21              5             fall  
4              19              0           spring  
(162541, 7)


In [3]:
movie_features = ratings.groupby('movieId').agg(
    movie_rating_count=('rating', 'count'),
    movie_avg_rating=('rating', 'mean'),
    movie_rating_std=('rating', 'std'),
    movie_peak_hour=('hour', lambda x: x.value_counts().idxmax()),
    movie_peak_season=('month', lambda x: (
        'winter' if x.value_counts().idxmax() in [12, 1, 2]
        else 'spring' if x.value_counts().idxmax() in [3, 4, 5]
        else 'summer' if x.value_counts().idxmax() in [6, 7, 8]
        else 'fall'
    ))
).reset_index()

print(movie_features.head())
print(movie_features.shape)

   movieId  movie_rating_count  movie_avg_rating  movie_rating_std  \
0        1               57309          3.893708          0.921552   
1        2               24228          3.251527          0.959851   
2        3               11804          3.142028          1.008443   
3        4                2523          2.853547          1.108531   
4        5               11714          3.058434          0.996611   

   movie_peak_hour movie_peak_season  
0               21              fall  
1               20              fall  
2               18            spring  
3               18              fall  
4               18            summer  
(59047, 6)


In [4]:
genres_split = movies['genres'].str.get_dummies('|')
print(genres_split.head())
print(genres_split.columns.tolist())

   (no genres listed)  Action  Adventure  Animation  Children  Comedy  Crime  \
0                   0       0          1          1         1       1      0   
1                   0       0          1          0         1       0      0   
2                   0       0          0          0         0       1      0   
3                   0       0          0          0         0       1      0   
4                   0       0          0          0         0       1      0   

   Documentary  Drama  Fantasy  Film-Noir  Horror  IMAX  Musical  Mystery  \
0            0      0        1          0       0     0        0        0   
1            0      0        1          0       0     0        0        0   
2            0      0        0          0       0     0        0        0   
3            0      1        0          0       0     0        0        0   
4            0      0        0          0       0     0        0        0   

   Romance  Sci-Fi  Thriller  War  Western  
0        0 

In [5]:
# Attach movieId to genres_split
genres_split['movieId'] = movies['movieId']

# Merge into movie_features
movie_features = movie_features.merge(genres_split, on='movieId')

print(movie_features.shape)
print(movie_features.head())

(59047, 26)
   movieId  movie_rating_count  movie_avg_rating  movie_rating_std  \
0        1               57309          3.893708          0.921552   
1        2               24228          3.251527          0.959851   
2        3               11804          3.142028          1.008443   
3        4                2523          2.853547          1.108531   
4        5               11714          3.058434          0.996611   

   movie_peak_hour movie_peak_season  (no genres listed)  Action  Adventure  \
0               21              fall                   0       0          1   
1               20              fall                   0       0          1   
2               18            spring                   0       0          0   
3               18              fall                   0       0          0   
4               18            summer                   0       0          0   

   Animation  ...  Film-Noir  Horror  IMAX  Musical  Mystery  Romance  Sci-Fi  \
0          

In [6]:
# Merge ratings with user features
dataset = ratings.merge(user_features, on='userId')

# Merge with movie features
dataset = dataset.merge(movie_features, on='movieId')

print(dataset.shape)
print(dataset.head())

(25000095, 40)
   userId  movieId  rating   timestamp            datetime  hour  day_of_week  \
0       1      296     5.0  1147880044 2006-05-17 15:34:04    15            2   
1       1      306     3.5  1147868817 2006-05-17 12:26:57    12            2   
2       1      307     5.0  1147868828 2006-05-17 12:27:08    12            2   
3       1      665     5.0  1147878820 2006-05-17 15:13:40    15            2   
4       1      899     3.5  1147868510 2006-05-17 12:21:50    12            2   

   month  year  user_rating_count  ...  Film-Noir  Horror  IMAX  Musical  \
0      5  2006                 70  ...          0       0     0        0   
1      5  2006                 70  ...          0       0     0        0   
2      5  2006                 70  ...          0       0     0        0   
3      5  2006                 70  ...          0       0     0        0   
4      5  2006                 70  ...          0       0     0        1   

  Mystery  Romance  Sci-Fi  Thriller  War

In [7]:
# Create binary target, watched = 1
dataset['watched'] = 1
print(dataset['watched'].value_counts())

watched
1    25000095
Name: count, dtype: int64


In [8]:
import random

all_users = dataset['userId'].unique()
all_movies = dataset['movieId'].unique()

existing_pairs = set(zip(dataset['userId'], dataset['movieId']))

negative_samples = []
n_negatives = len(dataset)

random.seed(42)
while len(negative_samples) < n_negatives:
    user = random.choice(all_users)
    movie = random.choice(all_movies)
    if (user, movie) not in existing_pairs:
        negative_samples.append((user, movie, 0))

print(f"Generated {len(negative_samples)} negative samples")

Generated 25000095 negative samples


In [9]:
# Convert negatives to dataframe
negatives_df = pd.DataFrame(negative_samples, columns=['userId', 'movieId', 'watched'])

# Add watched column to positives
dataset['watched'] = 1

# Keep only columns we need from positives
positives_df = dataset[['userId', 'movieId', 'watched']]

# Combine
full_dataset = pd.concat([positives_df, negatives_df], ignore_index=True)

print(full_dataset.shape)
print(full_dataset['watched'].value_counts())

(50000190, 3)
watched
1    25000095
0    25000095
Name: count, dtype: int64


In [10]:
# Merge user features
full_dataset = full_dataset.merge(user_features, on='userId')

# Merge movie features
full_dataset = full_dataset.merge(movie_features, on='movieId')

print(full_dataset.shape)
print(full_dataset.head())

(50000190, 34)
   userId  movieId  watched  user_rating_count  user_avg_rating  \
0       1      296        1                 70         3.814286   
1       1      306        1                 70         3.814286   
2       1      307        1                 70         3.814286   
3       1      665        1                 70         3.814286   
4       1      899        1                 70         3.814286   

   user_rating_std  user_peak_hour  user_peak_day user_peak_season  \
0         1.004235              12              2           spring   
1         1.004235              12              2           spring   
2         1.004235              12              2           spring   
3         1.004235              12              2           spring   
4         1.004235              12              2           spring   

   movie_rating_count  ...  Film-Noir  Horror  IMAX Musical  Mystery  Romance  \
0               79672  ...          0       0     0       0        0        0   

In [11]:
full_dataset.to_csv(f'{DATA_PATH}/full_dataset.csv', index=False)
print("Saved successfully")

Saved successfully


In [12]:
full_dataset.to_parquet(f'{DATA_PATH}/full_dataset.parquet', index=False)
print("Saved as parquet")

Saved as parquet
